In [0]:
dbutils.widgets.text("catalog","dev_l2")
catalog = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"use catalog {catalog}")
from pyspark.sql import functions as F

In [0]:
def valid_pk(col):
    return F.col(col).isNotNull()

def valid_date(col):
    return (F.col(col) >= F.to_date(F.lit('1992-01-01'))) & (F.col(col) <= F.to_date(F.lit('1998-12-31')))

def valid_fk(col,ref_col, ref):
    return (F.col(col).isNotNull()) & F.col(col).isin([row[ref_col] for row in ref.select(ref_col).distinct().collect()])

In [0]:
lineitem = spark.read.table("bronze.lineitem")
order = spark.read.table("silver.order")
part = spark.read.table("silver.part")
supplier = spark.read.table("silver.supplier")

In [0]:
lineitem_std = lineitem.select(
    "l_orderkey",
    "l_partkey",
    "l_suppkey",
    "l_linenumber",
    "l_quantity",
    "l_extendedprice",
    "l_discount",
    "l_tax",
    "l_returnflag",
    "l_linestatus",
    "l_shipdate",
    "l_commitdate",
    "l_receiptdate",
    F.trim(F.upper(F.col("l_shipinstruct"))).alias("l_shipinstruct"),
    F.trim(F.upper(F.col("l_shipmode"))).alias("l_shipmode"),
    F.trim(F.col("l_comment")).alias("l_comment")
).dropDuplicates()

In [0]:
valid_quantity = F.col("l_quantity") > 0
valid_extended_price = F.col("l_extendedprice") > 0
valid_discount = (F.col("l_discount") >= 0) & (F.col("l_discount") <= 0.1)
valid_tax = (F.col("l_tax") >= 0) & (F.col("l_tax") <= 0.08)
valid_receipt_date = valid_date("l_receiptdate") & (F.col("l_receiptdate") >= F.col("l_shipdate"))
valid_ship_date = valid_date("l_shipdate") & (F.col("l_shipdate") >= F.col("o_orderdate"))

In [0]:
valid_lineitem = lineitem_std.join(order.select(F.col("o_orderkey").alias("l_orderkey"), "o_orderdate"), "l_orderkey", "left") \
    .filter(F.col("o_orderdate").isNotNull() & valid_pk("l_linenumber") & valid_fk("l_partkey", "p_partkey", part) & valid_fk("l_suppkey", "s_suppkey", supplier) & valid_quantity & valid_extended_price & valid_discount & valid_tax & valid_receipt_date & valid_ship_date & valid_date("l_commitdate")) \
        .drop("o_orderdate")
valid_lineitem.write.mode("overwrite").saveAsTable("silver.lineitem")

In [0]:
reject_reasons = F.array_compact(
    F.array(
        F.when(~valid_fk("l_orderkey", "o_orderkey", order) | ~valid_pk("l_linenumber"), "INVLAID_PRIMARY_KEY"),
        F.when(~valid_fk("l_partkey", "p_partkey", part) | ~valid_fk("l_suppkey", "s_suppkey", supplier), "INVALID_REFERENCE_KEY"),
        F.when(~valid_quantity, "INVALID_QUANTITY"),
        F.when(~valid_extended_price, "INVALID_EXTENDED_PRICE"),
        F.when(~valid_discount, "INVALID_DISCOUNT"),
        F.when(~valid_tax, "INVALID_TAX"),
        F.when(~valid_receipt_date, "INVALID_RECEIPT_DATE"),
        F.when(~valid_ship_date, "INVALID_SHIP_DATE"),
        F.when(~valid_date("l_commitdate"), "INVALID_COMMIT_DATE")
    )
)

quarantine_lineitem = lineitem_std.withColumn("reject_reasons", reject_reasons) \
    .filter(F.size(F.col("reject_reasons")) > 0)
quarantine_lineitem.write.mode("overwrite").saveAsTable("silver.quarantine_lineitem")

In [0]:
%sql
create table dev_l2.silver.quarantine_lineitem as select *, cast(null as string) as reject_reasons from dev_l2.silver.lineitem
limit 0